In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import warnings

# Отключаем предупреждения
warnings.filterwarnings('ignore')

In [4]:
# Загрузка очищенных данных (результат EDA)
df = pd.read_csv('cleaned_data.csv')

In [5]:
# Удаление выбросов для целевой переменной IC50
Q1 = df['IC50, mM'].quantile(0.25)
Q3 = df['IC50, mM'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Оставляем только те строки, где IC50 не превышает верхнюю границу выбросов
df = df[df['IC50, mM'] <= upper_bound]
print(f"Размер данных после удаления выбросов IC50: {df.shape}")

Размер данных после удаления выбросов IC50: (807, 89)


In [6]:
# Убираем все целевые переменные и классы, чтобы модель не могла подсмотреть ответ
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI', 'IC50_class', 'CC50_class', 'SI_median_class', 'SI_8_class']
X = df.drop(columns=targets_to_drop, errors='ignore')
y = df['IC50, mM']

# Разбиваем данные на обучающую (80%) и тестовую (20%) выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Задаем словарь с моделями и параметрами для перебора
models = {
    "Ridge (Линейная регрессия)": (Ridge(), {'alpha': [100.0, 200.0]}),
    "Random Forest (Случайный лес)": (RandomForestRegressor(random_state=42), {'n_estimators': [50, 100], 'max_depth': [None, 10]}),
    "Gradient Boosting (Градиентный бустинг)": (GradientBoostingRegressor(random_state=42), {'n_estimators': [100, 300, 500], 'learning_rate': [0.01, 0.05, 0.1]}),
    "SVR (Опорные векторы)": (SVR(), {'C': [50.0, 100.0]}),
    "KNN (Ближайшие соседи)": (KNeighborsRegressor(), {'n_neighbors': [10, 20]})
    }

In [9]:
# Обучение и оценка моделей
results = {}

# Цикл по всем моделям
for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)

    # Предсказание на лучшей модели
    best_model = grid.best_estimator_
    predictions = best_model.predict(X_test_scaled)

    # Расчет метрик
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)

    results[name] = {'R2': r2, 'MAE': mae, 'Best Params': grid.best_params_}

In [10]:
# Итоговые результаты
print("Итоговые результаты сравнения моделей (IC50):\n")
for name, metrics in results.items():
    print(f"Модель: {name}")
    print(f"Лучшие параметры: {metrics['Best Params']}")
    print(f"Метрика R2: {metrics['R2']:.3f}")
    print(f"Ошибка MAE: {metrics['MAE']:.3f}\n")

Итоговые результаты сравнения моделей (IC50):

Модель: Ridge (Линейная регрессия)
Лучшие параметры: {'alpha': 200.0}
Метрика R2: 0.108
Ошибка MAE: 89.150

Модель: Random Forest (Случайный лес)
Лучшие параметры: {'max_depth': 10, 'n_estimators': 100}
Метрика R2: 0.123
Ошибка MAE: 83.335

Модель: Gradient Boosting (Градиентный бустинг)
Лучшие параметры: {'learning_rate': 0.01, 'n_estimators': 100}
Метрика R2: 0.113
Ошибка MAE: 88.837

Модель: SVR (Опорные векторы)
Лучшие параметры: {'C': 100.0}
Метрика R2: 0.196
Ошибка MAE: 68.086

Модель: KNN (Ближайшие соседи)
Лучшие параметры: {'n_neighbors': 20}
Метрика R2: 0.134
Ошибка MAE: 83.614



**Вывод для IC50.**

В ходе прогнозирования IC50 лучший результат показала модель SVR (R2 = 0.196, MAE = 68.1). Случайный лес, градиентный бустинг и KNN отработали примерно на одном уровне с R2 около 0.11-0.13, а линейная регрессия (Ridge) справилась хуже всего.

**Применимость.** Из-за очень низкого значения R2 (<0.2) можно сделать вывод, что применять эти регрессионные модели для точного предсказания концентрации IC50 на текущих дескрипторах нельзя, т.к. модель не улавливает закономерности.

**Рекомендации по улучшению:**
* расширить выборку новыми веществами;
* перейти от задачи регрессии к классификации (предсказывать категорию активности вместо точного числа).